In [11]:
# Import Pandas for data manipulation
import pandas as pd

print("Loading data...")

df = pd.read_csv('/Users/rajvardhansinghthakur/Documents/Inventory_Project/train.csv')
df.shape


Loading data...


(913000, 4)

In [12]:
df = df.sample(n=100000, random_state=42).copy()
df.shape

(100000, 4)

In [13]:
df.head()

,date,store,item,sales
491212,2013-01-19,10,27,21
64903,2015-09-21,6,4,15
36378,2017-08-12,10,2,100
133834,2014-06-21,4,8,93
633538,2017-10-09,7,35,33


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100000 entries, 491212 to 99850
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   date    100000 non-null  object
 1   store   100000 non-null  int64 
 2   item    100000 non-null  int64 
 3   sales   100000 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 3.8+ MB


In [15]:
df.isnull().sum()

date     0
store    0
item     0
sales    0
dtype: int64

In [16]:
print("Engineering features from the Date column...")

# Convert the text date into a Pandas Datetime format
df['date'] = pd.to_datetime(df['date'])

# Extract useful numbers for the Random Forest to learn from
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['dayofweek'] = df['date'].dt.dayofweek # Monday=0, Sunday=6

# Create a binary flag: 1 if it's the weekend, 0 if it's a weekday
df['is_weekend'] = df['dayofweek'].apply(lambda x: 1 if x >= 5 else 0)

# Drop the original 'date' column because the AI only understands numbers
df = df.drop('date', axis=1)

print("Feature Engineering Complete! Look at the new columns:")
df.head()

Engineering features from the Date column...
Feature Engineering Complete! Look at the new columns:


,store,item,sales,year,month,day,dayofweek,is_weekend
491212,10,27,21,2013,1,19,5,1
64903,6,4,15,2015,9,21,0,0
36378,10,2,100,2017,8,12,5,1
133834,4,8,93,2014,6,21,5,1
633538,7,35,33,2017,10,9,0,0


In [17]:
df.tail()

,store,item,sales,year,month,day,dayofweek,is_weekend
725134,8,40,33,2013,8,1,3,0
813627,6,45,63,2015,11,24,1,0
659854,2,37,41,2014,10,31,4,0
167332,2,10,112,2016,3,12,5,1
99850,5,6,48,2016,5,31,1,0


In [18]:
# Import the specific function to split data
from sklearn.model_selection import train_test_split

# X is the input data (features). y is the output we want to guess (sales)
X = df[['store', 'item', 'year', 'month', 'day', 'dayofweek', 'is_weekend']]
y = df['sales']

# Split 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training Data: {X_train.shape}")
print(f"Testing Data: {X_test.shape}")

Training Data: (80000, 7)
Testing Data: (20000, 7)


In [19]:
# Import the Random Forest model
from sklearn.ensemble import RandomForestRegressor

print("Training the Random Forest Model...")

# Initialize the model (50 trees, use all processor cores)
model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)

# The fit() command is where the AI actually learns the patterns
model.fit(X_train, y_train)

print("Model Training Complete!")

Training the Random Forest Model...
Model Training Complete!


In [20]:
# Import the tool to calculate the average error
from sklearn.metrics import mean_absolute_error

# Have the model take the test
predictions = model.predict(X_test)

# Calculate how wrong the model is on average
error = mean_absolute_error(y_test, predictions)
print(f"Average Error: The model is off by {round(error, 2)} units per prediction.")

Average Error: The model is off by 6.69 units per prediction.
